In [23]:
# ============================================================
# CELL 1 - PHISHGUARD AI FINAL SETUP
# ============================================================

from pathlib import Path
import sys
import json
import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 70)
print("PHISHGUARD AI - FINAL PREDICTION")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

PHISHGUARD AI - FINAL PREDICTION

Project root:
C:\Users\Manid\project1


In [24]:
# ============================================================
# CELL 2 - LOAD RANDOM FOREST MODEL
# ============================================================

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "random_forest_25_features.joblib"
)

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model not found:\n{MODEL_PATH}"
    )

rf_model = joblib.load(MODEL_PATH)

print("Random Forest loaded successfully. ✅")
print("Trees:", rf_model.n_estimators)
print("Features:", rf_model.n_features_in_)

Random Forest loaded successfully. ✅
Trees: 300
Features: 25


In [25]:
# ============================================================
# CELL 3 - LOAD FEATURE EXTRACTOR
# ============================================================

from src.features.url_features import (
    extract_url_features
)

print(
    "URL feature extractor loaded successfully. ✅"
)

URL feature extractor loaded successfully. ✅


In [26]:
# ============================================================
# CELL 4 - LOAD FEATURE COLUMN ORDER
# ============================================================

FEATURE_COLUMNS_PATH = (
    PROJECT_ROOT
    / "models"
    / "feature_columns.json"
)

if not FEATURE_COLUMNS_PATH.exists():
    raise FileNotFoundError(
        f"Feature column file not found:\n"
        f"{FEATURE_COLUMNS_PATH}"
    )

with open(
    FEATURE_COLUMNS_PATH,
    "r",
    encoding="utf-8"
) as file:

    FEATURE_COLUMNS = json.load(file)

print("Feature columns loaded. ✅")
print("Number of features:", len(FEATURE_COLUMNS))

if len(FEATURE_COLUMNS) != 25:
    raise RuntimeError(
        f"Expected 25 features, "
        f"found {len(FEATURE_COLUMNS)}."
    )

Feature columns loaded. ✅
Number of features: 25


In [27]:
# ============================================================
# CELL 5 - URL PREDICTION FUNCTION
# ============================================================

def predict_url(url):

    if not isinstance(url, str) or not url.strip():
        raise ValueError(
            "Please provide a valid URL."
        )

    # Extract the SAME 25 features used during training
    features = extract_url_features(url)

    # Create one-row DataFrame
    feature_df = pd.DataFrame(
        [features]
    )

    # Make sure all expected features exist
    missing_features = [
        feature
        for feature in FEATURE_COLUMNS
        if feature not in feature_df.columns
    ]

    if missing_features:
        raise ValueError(
            "Missing features:\n"
            + "\n".join(missing_features)
        )

    # Keep EXACT training order
    feature_df = feature_df[
        FEATURE_COLUMNS
    ]

    # Prediction
    prediction = rf_model.predict(
        feature_df
    )[0]

    probabilities = (
        rf_model.predict_proba(
            feature_df
        )[0]
    )

    # Determine probability safely
    classes = list(
        rf_model.classes_
    )

    phishing_probability = 0.0
    legitimate_probability = 0.0

    for class_value, probability in zip(
        classes,
        probabilities
    ):

        if int(class_value) == 1:
            phishing_probability = float(
                probability
            )
        else:
            legitimate_probability = float(
                probability
            )

    if int(prediction) == 1:
        result = "PHISHING"
    else:
        result = "LEGITIMATE"

    return {
        "url": url,
        "result": result,
        "prediction": int(prediction),
        "phishing_probability":
            phishing_probability,
        "legitimate_probability":
            legitimate_probability,
        "features": features
    }

print(
    "Prediction function created successfully. ✅"
)

Prediction function created successfully. ✅


In [28]:
# ============================================================
# CELL 6 - TEST PREDICTION
# ============================================================

TEST_URL = "https://www.google.com"

prediction_result = predict_url(
    TEST_URL
)

print("=" * 70)
print("PREDICTION TEST")
print("=" * 70)

print("\nURL:")
print(prediction_result["url"])

print("\nPrediction:")
print(prediction_result["result"])

print(
    "\nPhishing probability:",
    f"{prediction_result['phishing_probability'] * 100:.2f}%"
)

print(
    "Legitimate probability:",
    f"{prediction_result['legitimate_probability'] * 100:.2f}%"
)

print("\nPrediction completed successfully. ✅")

PREDICTION TEST

URL:
https://www.google.com

Prediction:
PHISHING

Phishing probability: 100.00%
Legitimate probability: 0.00%

Prediction completed successfully. ✅


In [29]:
# ============================================================
# CELL 7 - RISK ENGINE
# ============================================================

def calculate_risk(phishing_probability):
    """
    Convert phishing probability into a risk score
    and risk level.
    """

    # Convert probability (0-1) to score (0-100)
    risk_score = round(
        phishing_probability * 100,
        2
    )

    if risk_score >= 75:
        risk_level = "HIGH"
    elif risk_score >= 40:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    return risk_score, risk_level


print("Risk engine created successfully. ✅")

Risk engine created successfully. ✅


In [30]:
# ============================================================
# CELL 8 - TEST RISK ENGINE
# ============================================================

test_probability = prediction_result[
    "phishing_probability"
]

risk_score, risk_level = calculate_risk(
    test_probability
)

print("=" * 70)
print("RISK ENGINE TEST")
print("=" * 70)

print("\nPhishing probability:")
print(
    f"{test_probability * 100:.2f}%"
)

print("\nRisk score:")
print(
    risk_score
)

print("\nRisk level:")
print(
    risk_level
)

print("\nRisk engine test completed successfully. ✅")

RISK ENGINE TEST

Phishing probability:
100.00%

Risk score:
100.0

Risk level:
HIGH

Risk engine test completed successfully. ✅


In [31]:
# ============================================================
# CELL 9 - EXPLANATION GENERATOR
# ============================================================

def generate_explanation(
    url,
    features,
    result
):
    """
    Generate a simple explanation for the prediction.
    """

    reasons = []

    # Check common suspicious URL characteristics
    if features.get("has_ip_address", 0):
        reasons.append(
            "The URL uses an IP address."
        )

    if features.get("has_at_symbol", 0):
        reasons.append(
            "The URL contains an @ symbol."
        )

    if features.get("has_suspicious_double_slash", 0):
        reasons.append(
            "The URL contains suspicious double slashes."
        )

    if features.get("has_percent_encoding", 0):
        reasons.append(
            "The URL contains percent encoding."
        )

    if features.get("has_query", 0):
        reasons.append(
            "The URL contains query parameters."
        )

    if features.get("url_length", 0) > 100:
        reasons.append(
            "The URL is unusually long."
        )

    if features.get("hostname_length", 0) > 50:
        reasons.append(
            "The hostname is unusually long."
        )

    # --------------------------------------------------------
    # Result-specific explanation
    # --------------------------------------------------------

    if result == "PHISHING":

        if reasons:
            explanation = (
                "The URL was classified as phishing. "
                "Potential indicators include: "
                + " ".join(reasons)
            )
        else:
            explanation = (
                "The URL was classified as phishing "
                "based on the machine-learning model."
            )

    else:

        if reasons:
            explanation = (
                "The URL was classified as legitimate, "
                "although some URL characteristics were "
                "detected."
            )
        else:
            explanation = (
                "The URL was classified as legitimate "
                "by the machine-learning model."
            )

    return explanation


print(
    "Explanation generator created successfully. ✅"
)

Explanation generator created successfully. ✅


In [32]:
# ============================================================
# CELL 10 - FINAL URL ANALYSIS
# ============================================================

def analyze_url(url):

    prediction_result = predict_url(url)

    risk_score, risk_level = calculate_risk(
        prediction_result["phishing_probability"]
    )

    explanation = generate_explanation(
        prediction_result["url"],
        prediction_result["features"],
        prediction_result["result"]
    )

    return {
        "url": prediction_result["url"],
        "result": prediction_result["result"],
        "risk_score": risk_score,
        "risk_level": risk_level,
        "explanation": explanation,
        "phishing_probability":
            prediction_result["phishing_probability"],
        "legitimate_probability":
            prediction_result["legitimate_probability"],
        "features":
            prediction_result["features"]
    }


print(
    "Final analysis function created successfully. ✅"
)

Final analysis function created successfully. ✅


In [33]:
# ============================================================
# CELL 11 - FINAL PHISHGUARD TEST
# ============================================================

TEST_URL = "https://www.google.com"

analysis = analyze_url(
    TEST_URL
)

print("=" * 70)
print("PHISHGUARD AI - FINAL URL ANALYSIS")
print("=" * 70)

print("\nURL:")
print(analysis["url"])

print("\nPrediction:")
print(analysis["result"])

print("\nRisk Score:")
print(
    f"{analysis['risk_score']}/100"
)

print("\nRisk Level:")
print(analysis["risk_level"])

print("\nPhishing Probability:")
print(
    f"{analysis['phishing_probability'] * 100:.2f}%"
)

print("\nLegitimate Probability:")
print(
    f"{analysis['legitimate_probability'] * 100:.2f}%"
)

print("\nExplanation:")
print(analysis["explanation"])

print("\n" + "=" * 70)
print("PHISHGUARD ANALYSIS COMPLETED SUCCESSFULLY. ✅")
print("=" * 70)

PHISHGUARD AI - FINAL URL ANALYSIS

URL:
https://www.google.com

Prediction:
PHISHING

Risk Score:
100.0/100

Risk Level:
HIGH

Phishing Probability:
100.00%

Legitimate Probability:
0.00%

Explanation:
The URL was classified as phishing based on the machine-learning model.

PHISHGUARD ANALYSIS COMPLETED SUCCESSFULLY. ✅
